# 类型标注基础

学习目标：为数据和函数写出清楚的类型约定，并区分静态类型检查与运行时行为。

前置知识：变量赋值、函数参数与返回值、列表与元组、字典、条件判断和异常处理。

运行环境：Python 3.12。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

## 1 变量与函数标注

### 1.1 在变量旁说明预期类型

类型标注（type annotation）为编辑器和静态类型检查器提供类型信息。静态检查根据代码及其标注分析类型是否匹配；Python 执行程序时不会自动按标注校验数据。

变量名后用冒号写预期类型，再用等号赋值。下面 attempts: int = 3 中，attempts 是变量名，int 是标注，3 才是实际赋给变量的值。只写 attempts: int 不会给它初始值。

In [1]:
attempts: int = 3
task_name: str = "导入"

print(task_name, attempts)  # 导入 3
print(type(attempts).__name__)  # int：来自实际赋值的整数对象。

导入 3
int


### 1.2 标注参数与返回值

函数参数在名称后标注类型，参数列表后的 -> 表示返回值类型。标注描述调用约定，不会完成转换。

-> None 表示函数正常结束时返回 None，常用于只执行打印或修改操作的函数。不写 return，或者只写 return，也会返回 None。

函数与参数的调用规则保持不变；本章只补充类型约定，泛型和检查器操作在“泛型与静态类型检查”中展开。

In [2]:
def make_label(name: str, count: int) -> str:
    """生成包含名称和次数的标签。"""
    return f"{name}：{count} 次"


def show_notice(message: str) -> None:
    """打印一条通知。"""
    print(message)  # 下面传入“已完成”，原样打印该文本。


print(make_label("练习", 3))  # 练习：3 次
returned = show_notice("已完成")  # 已完成
print(returned)  # None：没有 return 语句也有返回值。

练习：3 次
已完成
None


## 2 容器中的元素类型

list 和 dict 等内置容器可以直接用方括号标注元素类型，不需要导入 typing.List 或 typing.Dict。容器操作沿用已有规则，这里说明其中元素的类型约定。

| 标注写法 | 中文名称／含义 |
| --- | --- |
| list[str] | 字符串列表，每个元素都是字符串 |
| dict[str, int] | 字典，键为字符串，值为整数 |
| tuple[str, int] | 固定两项的元组，依次为字符串、整数 |
| tuple[int] | 只有一项的元组，该项为整数 |
| tuple[int, ...] | 任意长度的整数元组，也允许空元组 |

tuple[int, ...] 中的 ... 是省略号，表示长度不固定；它不是等待补全的代码。tuple[str, int] 按位置逐项约定，不能理解成任意长度的字符串或整数混合元组。

In [3]:
names: list[str] = ["林", "陈"]
scores: dict[str, int] = {"林": 80, "陈": 90}
record: tuple[str, int] = ("林", 80)
single_score: tuple[int] = (80,)
score_batch: tuple[int, ...] = ()

print(names[0], scores["林"])  # 林 80
print(record, single_score)  # ('林', 80) (80,)
print(score_batch)  # ()：任意长度也包含零项。

score_batch = (80, 90, 100)
print(score_batch)  # (80, 90, 100)：每项仍满足整数标注。

林 80
('林', 80) (80,)
()
(80, 90, 100)


## 3 用联合类型表达多种输入

联合类型（union type）用 | 连接允许的类型。int | str 表示整数或字符串，旧写法是 typing.Union[int, str]。这里的 | 连接类型，不是对整数值做按位或运算。

联合类型不会把一种输入自动变成另一种。下面 parse_count 明确调用 int 完成转换；字符串属于允许的类型，也不代表其内容一定能转换成整数。

In [4]:
def parse_count(value: int | str) -> int:
    """把整数或十进制整数文本转换为整数。"""
    return int(value)


print(parse_count(12), parse_count("12"))  # 12 12
print(parse_count("  -3 "))  # -3：转换规则来自 int，不是联合类型。

12 12
-3


In [5]:
# 预期 ValueError：直接观察原始异常，之后继续运行下一单元。
# ValueError：字符串内容无法转换。
parse_count("many")

ValueError: invalid literal for int() with base 10: 'many'

## 4 Optional 与参数默认值

Optional[T] 等价于 T | None，其中 T 表示某个类型，例如 str。两种写法都表示值可以是该类型的实例，也可以是 None；本章主要使用 | None。

是否允许 None 与调用时能否省略实参是两件事。name: str | None 没有默认值，仍须传入实参；name: str = "访客" 有默认值，但类型约定并不允许 None。写成 name: str | None = None 才同时表达两者。

判断缺失值时使用 is None，避免把合法的空字符串误判为缺失。

In [6]:
from typing import Optional


def display_name(name: str | None) -> str:
    """把缺失姓名显示为访客，保留已有字符串。"""
    return "访客" if name is None else name


def greet(name: str = "访客") -> str:
    """为姓名生成问候语，省略实参时使用默认姓名。"""
    return f"你好，{name}"


print(Optional[str] == (str | None))  # True：两种类型写法等价。
print(display_name(None), repr(display_name("")))  # 访客 ''
print(greet())  # 你好，访客：能省略实参是因为定义了默认值。

True
访客 ''
你好，访客


In [7]:
# 预期 TypeError：直接观察原始异常，之后继续运行下一单元。
# TypeError：可空参数依然缺少实参。
display_name()

TypeError: display_name() missing 1 required positional argument: 'name'

## 5 Any 与 object

Any 用于需要保留动态行为的接口，会放宽静态检查。把 Any 值赋给更具体的类型，检查器不会因此确认它在运行时确实具有那个类型。

| 名称 | 中文名称／含义 | 静态检查中的区别 |
| --- | --- | --- |
| typing.Any | 任意类型，表示动态类型行为 | 允许任意操作，并与其他类型相容 |
| object | 所有 Python 对象的基类 | 可接收任何对象，但不能据此假定对象有字符串等专属操作 |

如果函数能处理任意对象，又希望保留检查约束，可以标注 object。需要调用特定类型的方法时，应先确认类型；相关的类型收窄在下一章展开。

In [8]:
from typing import Any


def to_text(value: object) -> str:
    """取得任意对象的字符串表示。"""
    return str(value)


dynamic_value: Any = 42
text: str = dynamic_value
print(type(text).__name__)  # int：Any 到 str 的赋值没有转换数据。
print(to_text(42), to_text("Python"))  # 42 Python

int
42 Python


In [9]:
# 预期 AttributeError：直接观察原始异常，之后继续运行下一单元。
# AttributeError：实际的整数没有 upper。
text.upper()

AttributeError: 'int' object has no attribute 'upper'

## 6 Literal 限定具体的值

字面值类型（literal type）用 Literal 列出允许的具体值。Literal["csv", "json"] 表示静态检查时只接受这两个字符串值，比 str 的范围更小。

Literal 描述静态约定，不会自动检查函数实参。本章按给定类型约定使用输入，不额外添加运行时校验；下面的反例专门观察标注与执行行为的区别。

In [10]:
from typing import Literal


def make_filename(stem: str, extension: Literal["csv", "json"]) -> str:
    """拼接文件名，约定扩展名为 csv 或 json。"""
    return f"{stem}.{extension}"


print(make_filename("report", "csv"))  # report.csv

# 教学反例：违反 Literal 的静态约定，解释器仍会执行函数体。
print(make_filename("report", "txt"))  # report.txt：标注没有执行值校验。

report.csv
report.txt


## 7 类型别名与 type 语句

### 7.1 为重复的类型表达式命名

类型别名（type alias）为一个类型表达式提供另一个名称，适合复用有明确含义的类型。静态检查时，别名与它代表的类型等价，不会新建子类。

简单赋值可以定义类型别名。Python 3.12 新增的 type 语句则显式声明别名，运行时创建 typing.TypeAliasType 对象。下面两种写法都表示字符串键、整数值的字典。

type 语句的右侧延迟到访问别名的 \_\_value\_\_ 属性时求值；前向引用与带类型参数的用法留到下一章。

In [11]:
LegacyScores = dict[str, int]
type Scores = dict[str, int]


def total_score(grades: Scores) -> int:
    """求所有成绩之和，空字典返回零。"""
    return sum(grades.values())


legacy_grades: LegacyScores = {"林": 80, "陈": 90}
print(total_score(legacy_grades))  # 170：普通字典满足这个别名的约定。
print(total_score({}))  # 0

print(type(Scores).__name__)  # TypeAliasType：type 语句创建别名对象。
print(Scores.__value__ == dict[str, int])  # True：查看其代表的类型。

170
0
TypeAliasType
True


### 7.2 别名对象不充当构造器或类型检查目标

普通赋值直接绑定右侧对象。Count = int 使 Count 指向 int 类，因此 Count("3") 仍是调用 int。

type Quantity = int 创建的是别名对象，不是 int 类，也不是其子类。它不能当作构造器调用，也不能直接作为 isinstance 的第二个实参。需要创建或检查整数时，使用实际的 int 类。

In [12]:
Count = int
type Quantity = int

print(Count is int, Count("3"))  # True 3：普通赋值保留原对象。
quantity: Quantity = 3
print(isinstance(quantity, int))  # True：实际值是整数。

True 3
True


In [13]:
# 预期 TypeError：直接观察原始异常，之后继续运行下一单元。
# TypeError：别名对象不可调用。
Quantity("3")

TypeError: 'typing.TypeAliasType' object is not callable

In [14]:
# 预期 TypeError：直接观察原始异常，之后继续运行下一单元。
# TypeError：别名对象不是运行时类。
isinstance(quantity, Quantity)

TypeError: isinstance() arg 2 must be a type, a tuple of types, or a union

## 8 静态约定与运行时检查

类型标注既不会验证输入，也不会改变函数体的运算。下面反例故意违反标注：字符串仍执行重复操作，列表也仍能接收字符串。

isinstance 检查实际对象与运行时类的关系。isinstance(counts, list) 只能确认 counts 是列表，不能确认元素类型；list[int] 这样的带类型参数的标注不能直接作为其第二个实参。检查元素需要显式遍历。

本章输出来自 Python 执行；类型检查器的诊断需要另行运行工具，不应把程序运行成功当作静态检查通过。

In [15]:
def double_count(count: int) -> int:
    """把次数乘以二。"""
    return count * 2


print(double_count(3))  # 6
print(double_count("3"))  # 33：反例，字符串重复；没有变成整数。

counts: list[int] = [1, 2]
counts.append("3")  # 反例：违反元素类型约定，运行时并不阻止。
print(counts)  # [1, 2, '3']
print(isinstance(counts, list))  # True：只检查了外层容器。
print(all(isinstance(count, int) for count in counts))  # False

6
33
[1, 2, '3']
True
False


In [16]:
# 预期 TypeError：直接观察原始异常，之后继续运行下一单元。
# TypeError：不能用它逐项检查元素。
isinstance(counts, list[int])

TypeError: isinstance() argument 2 cannot be a parameterized generic

## 本章小结

（1）变量、参数和返回值标注表达预期类型；运行时的值来自实际赋值、传参和函数执行。

（2）容器标注描述元素类型；元组可逐项固定类型，也可用省略号表示同类型的任意长度。

（3）联合类型允许多种类型；Optional 表示允许 None，默认值决定能否省略实参。

（4）Any 放宽静态检查，object 保留可用操作的约束；Literal 限定静态允许的具体值。

（5）类型别名复用类型含义。type 语句创建的别名对象不能直接构造值，也不能直接传给 isinstance。

自查：能否分别说明一个调用是否满足类型约定、运行时是否成功，以及是否需要另外校验数据？

## 练习

（1）先预测下面三行输出，再执行核对。逐项说明省略实参、显式传入 None、传入空字符串时分别使用哪个值；再判断这三次调用是否都符合标注。

In [17]:
def choose_title(title: str | None = "草稿") -> str:
    """为缺失标题提供显示文本。"""
    return "未命名" if title is None else title


print(repr(choose_title()))
print(repr(choose_title(None)))
print(repr(choose_title("")))
# 核对时区分默认值与缺失值分支，不把空字符串当作 None。

'草稿'
'未命名'
''


（2）实现 first_passing(grades, minimum)，为参数及返回值补齐标注。grades 是整数成绩列表，minimum 是整数阈值；按原顺序返回第一个达到阈值的成绩，没有匹配项则返回 None。

用 [50, 80, 90]，分别把阈值设为 60、85、100；再用 [0] 和阈值 0。区分合法的零成绩与没有匹配结果。

In [18]:
exercise_grades: list[int] = [50, 80, 90]

# 定义 first_passing，并检查阈值 60、85、100 分别得到 80、90、None。
# [0] 配合阈值 0 应得到 0，不能把它误当作 None。
# 分支用于真实筛选；输入均为整数，不添加类型或范围防护。

（3）用 type 语句定义 ReportFormat，代表 Literal["csv", "json"]，并实现 build_report_name(stem, extension)，使用该别名标注 extension，默认值为 "csv"，返回字符串。

直接拼接合法的扩展名，检查默认值和显式 "json"。结合本章 Literal 反例解释：为什么标注能描述允许的值，却不会自动执行转换或校验？

In [19]:
report_stem = "成绩"

# 定义类型别名和函数。
# build_report_name(report_stem) 得到 "成绩.csv"。
# 显式传入 "json" 得到 "成绩.json"。
# 不调用 ReportFormat 转换扩展名，不增加成员检查。

### 提示

第一题分别判断默认实参与 is None 分支。第二题在循环内发现匹配就返回成绩，循环结束返回 None。第三题把别名用于标注，函数体直接生成文件名。

### 参考解析

第一题依次输出 '草稿'、'未命名'、''，三次调用都符合 str | None；省略实参采用默认值，None 触发缺失标题分支，空字符串原样保留。

第二题标注为 first_passing(grades: list[int], minimum: int) -> int | None。遍历成绩，用 grade >= minimum 决定返回；未找到时返回 None。阈值 60、85、100 分别得到 80、90、None；[0] 配合阈值 0 得到 0。

第三题声明 type ReportFormat = Literal["csv", "json"]，标注 extension: ReportFormat = "csv"，函数体返回 f"{stem}.{extension}"。默认和 json 两种调用分别得到“成绩.csv”和“成绩.json”；别名复用静态约定，运行时仍按函数体拼接字符串。

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| Python 官方文档（3.12） | [标注赋值语句](https://docs.python.org/3.12/reference/simple_stmts.html#annotated-assignment-statements)；[函数标注](https://docs.python.org/3.12/tutorial/controlflow.html#function-annotations)、[函数与 None 返回值](https://docs.python.org/3.12/tutorial/controlflow.html#defining-functions)、[默认参数值](https://docs.python.org/3.12/tutorial/controlflow.html#default-argument-values)；[typing：标注不由运行时强制执行](https://docs.python.org/3.12/library/typing.html)、[元组标注](https://docs.python.org/3.12/library/typing.html#annotating-tuples)、[Optional](https://docs.python.org/3.12/library/typing.html#typing.Optional)、[Any 与 object](https://docs.python.org/3.12/library/typing.html#the-any-type)、[Literal](https://docs.python.org/3.12/library/typing.html#typing.Literal)、[类型别名](https://docs.python.org/3.12/library/typing.html#type-aliases)、[TypeAliasType 与延迟求值](https://docs.python.org/3.12/library/typing.html#typing.TypeAliasType)；[type 语句](https://docs.python.org/3.12/reference/simple_stmts.html#the-type-statement)；[内置容器的参数化标注及运行时限制](https://docs.python.org/3.12/library/stdtypes.html#types-genericalias)、[联合类型](https://docs.python.org/3.12/library/stdtypes.html#types-union)、[序列重复与成员检查](https://docs.python.org/3.12/library/stdtypes.html#common-sequence-operations)、[dict.values](https://docs.python.org/3.12/library/stdtypes.html#dict.values)；[int 转换](https://docs.python.org/3.12/library/functions.html#int)、[isinstance](https://docs.python.org/3.12/library/functions.html#isinstance)、[all](https://docs.python.org/3.12/library/functions.html#all)、[sum](https://docs.python.org/3.12/library/functions.html#sum)；[ValueError](https://docs.python.org/3.12/library/exceptions.html#ValueError)、[AttributeError](https://docs.python.org/3.12/library/exceptions.html#AttributeError)、[TypeError](https://docs.python.org/3.12/library/exceptions.html#TypeError)。 |
| GitHub：CPython 官方源码（v3.12.14） | [Objects/typevarobject.c，第 1497–1515 行](https://github.com/python/cpython/blob/v3.12.14/Objects/typevarobject.c#L1497-L1515)：TypeAliasType 的运行时类型定义，未提供调用槽；用于核对 type 别名对象不能直接调用的实现依据。 |